# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
from udaplay.agent import (Agent, retrieve_game, evaluate_retrieval, game_web_search, EvaluationReport)

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
docs = retrieve_game("first 3D mario platformer")
docs[:5]  # Show first 5 results

[{'Name': 'Super Mario 64',
  'Platform': 'Nintendo 64',
  'YearOfRelease': 1996,
  'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.",
  'type': 'leaf',
  'level': 0,
  'score': 0.890768},
 {'Name': 'Super Mario 3D All-Stars',
  'Platform': 'Nintendo Switch',
  'YearOfRelease': 2020,
  'Description': 'Play three of Mario’s greatest 3D platform adventures—all in one package!\nPlay three classic games at home or on the go—all in one package on the Nintendo Switch™ system! Jump into paintings in Super Mario 64™, clean up paint-like goop in Super Mario Sunshine™, and fly from planet to planet in Super Mario Galaxy™.\n\nRun, jump, and dive with ease!\n\nMake Mario move using the Nintendo Switch system’s Joy-Con™ controllers. You can also pass a Joy-Con controller to a friend to play the Super Mario Galaxy game in Co-Star Mode! Mario’s movements are as smooth as ever with HD resolution for each game, while 

#### Evaluate Retrieval Tool

In [5]:
evaluate_retrieval("first 3D mario platformer", docs).model_dump()

{'useful': True,
 'description': "The documents provide information about 'Super Mario 64', which is identified as the first 3D Mario platformer, released in 1996."}

#### Game Web Search Tool

In [6]:
from udaplay.index_builder import get_chroma_client, LONGTERM
try:
    get_chroma_client().delete_collection(LONGTERM)
except Exception:
    pass
print("Deleted existing collection if it existed.")
game_web_search("Was Mortal Kombat X released for PlayStation 5?")["answer"]

Deleted existing collection if it existed.


'Mortal Kombat X was not released for PlayStation 5. It was originally released for PlayStation 4 in 2015. It will be part of the PlayStation Plus collection for PS5 at launch.'

### Agent

In [7]:
def print_trace(messages):
    """Print one turn's messages: tool calls by name, tool results, assistant text."""
    for m in messages:
        role = m.get("role")
        if role == "user":
            print(f"USER: {m['content']}")
        elif role == "assistant":
            for tc in (m.get("tool_calls") or []):
                fn = tc["function"]
                print(f"  -> call {fn['name']}({fn['arguments']})")
            if m.get("content"):
                print(f"ASSISTANT: {m['content']}")
        elif role == "tool":
            content = m.get("content", "")
            snippet = content[:200] + ("…" if len(content) > 200 else "")
            print(f"  <- {m.get('name')} returned: {snippet}")

agent = Agent()

In [8]:
questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

seen = 0
for q in questions:
    ans = agent.invoke(q, session_id="report")
    msgs = agent.sessions["report"]
    print(f"\n===== {q} =====")
    print_trace(msgs[seen:])
    seen = len(msgs)
    print("FINAL:", ans.answer)
    print("citations:", ans.citations, "| source:", ans.source,
          "| confidence:", ans.confidence)
    print("-" * 70)
    follow_up_question = "What platform was that game on?"
    ans = agent.invoke(follow_up_question, session_id="report")
    msgs = agent.sessions["report"]
    print(f"\n===== {follow_up_question} =====")
    print_trace(msgs[seen:])
    seen = len(msgs)


===== When was Pokémon Gold and Silver released? =====
USER: When was Pokémon Gold and Silver released?
  -> call retrieve_game({"query":"Pokémon Gold and Silver release date"})
  <- retrieve_game returned: [{"Name": "Pok\u00e9mon Gold and Silver", "Platform": "Game Boy Color", "YearOfRelease": 1999, "Description": "Second-generation Pok\u00e9mon games introducing new regions, Pok\u00e9mon, and gameplay …
  -> call evaluate_retrieval({"question":"When was Pokémon Gold and Silver released?","retrieved_docs":[{"Name":"Pokémon Gold and Silver","Platform":"Game Boy Color","YearOfRelease":1999,"Description":"Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics."},{"Name":null,"Platform":null,"YearOfRelease":null,"Description":"The records highlight a range of video games primarily published by Nintendo, spanning various genres including role-playing games (RPGs) and simulation games, across different platforms such as the Game Boy Color, Game Boy Advance

### (Optional) Advanced

In [9]:
agent.invoke("Who developed Hades?", session_id="mem-demo")          # web -> stored
agent.memory.search("Hades developer", k=1) 

[{'content': 'Q: Was Mortal Kombat X released for PlayStation 5?\nA: Mortal Kombat X was not released for PlayStation 5. It was originally released for PlayStation 4 in 2015. It will be part of the PlayStation Plus collection for PS5 at launch.',
  'source': 'https://www.gamestop.com/video-games/playstation-5?q=%22Mortal+Kombat+X%22',
  'score': 0.712503}]

In [10]:
from udaplay.eval_harness import run_ablation
from udaplay.visualize import draw_ablation
df = run_ablation()
display(df)
draw_ablation(df, save_path="ablation.png")

,hit_rate,mean_top_score,n_queries
udaplay_baseline,1.0,0.835253,12.0
udaplay_raptor,1.0,0.841798,12.0
udaplay_raptor_sac,1.0,0.841985,12.0


<Axes: title={'center': 'Ablation: Baseline vs RAPTOR vs RAPTOR+SAC'}, ylabel='score'>